<a href="https://colab.research.google.com/github/Ruchi74/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Ruchi74/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

### Signal Check 1: Content freshness and visibility

**Verdict: MIXED**

Only 17 pages were both stale (not updated for at least 180 days) and visible (at least 500 impressions). Their median impressions were 4,429, suggesting that some stale pages still receive visibility. However, the small sample size means this signal needs further validation before using it alone.

### Signal Check 2: Low CTR on visible pages

**Verdict: CONFIRMED**

Among pages with at least 500 impressions and an average position from 1 to 20, 9,759 pages had CTR below 0.5%, compared with 2,264 pages with CTR of at least 0.5%. Median CTR was 0.17% for the low-CTR group and 0.73% for the comparison group. This supports investigating low-CTR pages as possible CTR-improvement opportunities, but does not prove that making changes will improve performance.

In [4]:

import pandas as pd
from pathlib import Path

# Load the dataset
queue = pd.read_csv(data_url)

# Rule: identify visible pages with low CTR
eligible = (
    (queue["impressions_90d"] >= 500) &
    (queue["avg_position"] > 0) &
    (queue["avg_position"] <= 20) &
    (queue["ctr"] < 0.5)
)

# Assign score, reason code, and action
queue["score"] = eligible.astype(int)
queue["reason_code"] = "NO_ACTION"
queue["action"] = "MONITOR"

queue.loc[eligible, "reason_code"] = "LOW_CTR_VISIBLE_PAGE"
queue.loc[eligible, "action"] = "REVIEW_CTR"

# Rank all rows: eligible pages first, then by impressions
queue = queue.sort_values(
    ["score", "impressions_90d"],
    ascending=[False, False]
).reset_index(drop=True)

queue["rank"] = range(1, len(queue) + 1)

# Save the ranked queue
output_dir = Path("work/outputs")
output_dir.mkdir(parents=True, exist_ok=True)

output_path = output_dir / "baseline_action_score.csv"
queue.to_csv(output_path, index=False)

print("Ranked queue created successfully!")
print("Total rows:", len(queue))
print("Pages recommended for CTR review:", int(eligible.sum()))
print("CSV saved at:", output_path)

print("\nTop 10 items:")
print(queue[
    ["rank", "content_id", "score", "reason_code",
     "action", "impressions_90d", "ctr", "avg_position"]
].head(10).to_string(index=False))


Ranked queue created successfully!
Total rows: 30000
Pages recommended for CTR review: 9759
CSV saved at: work/outputs/baseline_action_score.csv

Top 10 items:
 rank           content_id  score          reason_code     action  impressions_90d  ctr  avg_position
    1 content_5fe46e04994d      1 LOW_CTR_VISIBLE_PAGE REVIEW_CTR           517715 0.14           4.2
    2 content_aaef01a50def      1 LOW_CTR_VISIBLE_PAGE REVIEW_CTR           517109 0.25           5.4
    3 content_8c19996aa890      1 LOW_CTR_VISIBLE_PAGE REVIEW_CTR           509252 0.15           2.5
    4 content_4c36c775b818      1 LOW_CTR_VISIBLE_PAGE REVIEW_CTR           463103 0.41           2.3
    5 content_1a9e894be2e2      1 LOW_CTR_VISIBLE_PAGE REVIEW_CTR           416180 0.23           4.0
    6 content_db5989a78dd3      1 LOW_CTR_VISIBLE_PAGE REVIEW_CTR           345111 0.21           5.4
    7 content_cb112fce36be      1 LOW_CTR_VISIBLE_PAGE REVIEW_CTR           309910 0.16           5.6
    8 content_36ff89c821

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


In [6]:
import pandas as pd
from pathlib import Path

data_url = (
    "https://raw.githubusercontent.com/"
    "Ruchi74/flyrank-ml-internship/main/"
    "data/raw/content_refresh_anonymized.csv"
)

df = pd.read_csv(data_url)

print("Dataset loaded successfully")
print("Rows and columns:", df.shape)
print("\nColumn names:")
print(df.columns.tolist())

Dataset loaded successfully
Rows and columns: (30000, 44)

Column names:
['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct']


In [7]:

# WEEK 4 — SIGNAL CHECKS
import pandas as pd
import numpy as np

# Signal 1: Content freshness and visibility
df["stale_flag"] = df["days_since_last_update"] >= 180
df["visible_flag"] = df["impressions_90d"] >= 500

df["signal_1_bucket"] = np.select(
    [
        df["stale_flag"] & df["visible_flag"],
        ~df["stale_flag"] & df["visible_flag"],
        df["stale_flag"] & ~df["visible_flag"]
    ],
    [
        "Stale + visible",
        "Fresh + visible",
        "Stale + low visibility"
    ],
    default="Fresh + low visibility"
)

print("SIGNAL 1: Freshness and visibility")
print(
    df.groupby("signal_1_bucket", observed=True)
      .agg(
          n=("content_id", "size"),
          median_impressions=("impressions_90d", "median"),
          median_days_since_update=("days_since_last_update", "median")
      )
      .reset_index()
      .to_string(index=False)
)

# Signal 2: Low CTR among visible pages with known positions
valid_position = (
    (df["avg_position"] > 0) &
    (df["avg_position"] <= 20) &
    (df["impressions_90d"] >= 500)
)

df["signal_2_bucket"] = "Other"
df.loc[valid_position & (df["ctr"] < 0.5), "signal_2_bucket"] = "Low CTR"
df.loc[valid_position & (df["ctr"] >= 0.5), "signal_2_bucket"] = "CTR at least 0.5%"

print("\nSIGNAL 2: CTR among visible pages ranked in positions 1–20")
print(
    df.loc[valid_position]
      .groupby("signal_2_bucket", observed=True)
      .agg(
          n=("content_id", "size"),
          median_ctr=("ctr", "median"),
          median_impressions=("impressions_90d", "median"),
          median_position=("avg_position", "median")
      )
      .reset_index()
      .to_string(index=False)
)


SIGNAL 1: Freshness and visibility
       signal_1_bucket     n  median_impressions  median_days_since_update
Fresh + low visibility 13117                55.0                      20.0
       Fresh + visible 16709              2948.0                      22.0
Stale + low visibility   157                13.0                     211.0
       Stale + visible    17              4429.0                     194.0

SIGNAL 2: CTR among visible pages ranked in positions 1–20
  signal_2_bucket    n  median_ctr  median_impressions  median_position
CTR at least 0.5% 2264        0.73              4729.5              7.2
          Low CTR 9759        0.17              3017.0              8.4


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

### Top-10 Review

The baseline ranks pages that have at least 500 impressions in the last 90 days, an average position from 1 to 20, and CTR below 0.5%. The ranking prioritizes higher impressions. These are candidates for review, not guaranteed improvements.

| Rank | Action | Why it is selected | What could make it wrong |
|---:|---|---|---|
| 1 | REVIEW_CTR | CTR 0.14%, 517,715 impressions, position 4.2 | The title and snippet may already accurately match the search intent. |
| 2 | REVIEW_CTR | CTR 0.25%, 517,109 impressions, position 5.4 | Search-result features or query mix may explain the CTR. |
| 3 | REVIEW_CTR | CTR 0.15%, 509,252 impressions, position 2.5 | The page may already attract the appropriate audience. |
| 4 | REVIEW_CTR | CTR 0.41%, 463,103 impressions, position 2.3 | CTR is below the rule threshold but may be reasonable for its queries. |
| 5 | REVIEW_CTR | CTR 0.23%, 416,180 impressions, position 4.0 | The impressions may come from queries with different intent. |
| 6 | REVIEW_CTR | CTR 0.21%, 345,111 impressions, position 5.4 | The page may need a ranking improvement rather than a title change. |
| 7 | REVIEW_CTR | CTR 0.16%, 309,910 impressions, position 5.6 | Search-result layout may reduce clicks regardless of the page title. |
| 8 | REVIEW_CTR | CTR 0.05%, 295,097 impressions, position 7.3 | The data may combine unrelated queries or search intents. |
| 9 | REVIEW_CTR | CTR 0.03%, 272,144 impressions, position 2.3 | The page could rank for broad or poorly matched queries. |
| 10 | REVIEW_CTR | CTR 0.26%, 236,803 impressions, position 4.4 | The CTR difference may reflect the query mix rather than a page problem. |

**Review conclusion:** All ten rows meet the baseline rule, but they are not confirmed opportunities. Before taking action, inspect query-level data, search intent, and existing

In [9]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

### Weak Picks + Leakage Check

**Weak picks:** Some pages may have low CTR because of their query mix, search-result features, or search intent rather than an incorrect title or snippet. Pages with average positions near the bottom of the eligible range may need ranking improvements instead of CTR changes. The rule is a triage method, not proof of a problem.

**Leakage check:** The baseline uses only `impressions_90d`, `avg_position`, and `ctr` from the supplied dataset. It does not use `trend_direction`, `trend_pct`, `is_declining_label`, future-window measurements, or product decision labels as scoring inputs.

**Limitation:** The baseline has a transparent rule score, not a trained prediction model. Its effectiveness has not been validated against future outcomes.

In [10]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.